# DX 704 Week 4 Live Session

Note: this similar to the homework, but not solving the exact same problems.

In [ ]:
import numpy as np
import pandas as pd

## Load Data

In [19]:
recipes = pd.read_csv('https://raw.githubusercontent.com/bu-cds-dx704/dx704-project-04/refs/heads/main/recipes.tsv', sep="\t")
recipes = recipes.set_index("recipe_slug")
recipes

,recipe_title,recipe_introduction
recipe_slug,,
falafel,Falafel,Falafel is a popular Middle Eastern dish made ...
spamburger,Spamburger,Spamburger is a type of hamburger that is made...
bacon-fried-rice,Bacon Fried Rice,Bacon fried rice is a savory and satisfying di...
chicken-fingers,Chicken Fingers,Chicken fingers are a popular dish made from c...
apple-crisp,Apple Crisp,Apple crisp is a classic dessert made with bak...
...,...,...
bacon-mac-and-cheese,Bacon Mac And Cheese,Bacon mac and cheese is a delicious and comfor...
chicken-alfredo-lasagna,Chicken Alfredo Lasagna,Chicken alfredo lasagna is a delicious twist o...
classic-beef-lasagna,Classic Beef Lasagna,Classic beef lasagna is a hearty and comfortin...


## Class Feature Choice

In [20]:
def add_feature(feature_name):
    recipes[feature_name] = recipes["recipe_title"].str.lower().str.contains(feature_name.lower())

In [21]:
add_feature("apple")
add_feature("bacon")
add_feature("chicken")
add_feature("crisp")
add_feature("lasagna")
add_feature("veg")

recipes

,recipe_title,recipe_introduction,apple,bacon,chicken,crisp,lasagna,veg
recipe_slug,,,,,,,,
falafel,Falafel,Falafel is a popular Middle Eastern dish made ...,False,False,False,False,False,False
spamburger,Spamburger,Spamburger is a type of hamburger that is made...,False,False,False,False,False,False
bacon-fried-rice,Bacon Fried Rice,Bacon fried rice is a savory and satisfying di...,False,True,False,False,False,False
chicken-fingers,Chicken Fingers,Chicken fingers are a popular dish made from c...,False,False,True,False,False,False
apple-crisp,Apple Crisp,Apple crisp is a classic dessert made with bak...,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...
bacon-mac-and-cheese,Bacon Mac And Cheese,Bacon mac and cheese is a delicious and comfor...,False,True,False,False,False,False
chicken-alfredo-lasagna,Chicken Alfredo Lasagna,Chicken alfredo lasagna is a delicious twist o...,False,False,True,False,True,False
classic-beef-lasagna,Classic Beef Lasagna,Classic beef lasagna is a hearty and comfortin...,False,False,False,False,True,False


## Class Choice for Ratings

In [22]:
ratings = pd.DataFrame({"recipe_title": recipes["recipe_title"],
                               "rating_bad": np.random.uniform(low=2, high=3, size=len(recipes))},
                              index=recipes.index)
ratings.loc[recipes['veg'], 'rating_bad'] = 1
ratings.loc[recipes['bacon'], 'rating_bad'] = 5
ratings['rating'] = ratings['rating_bad']
ratings

,recipe_title,rating_bad,rating
recipe_slug,,,
falafel,Falafel,2.809655,2.809655
spamburger,Spamburger,2.304000,2.304000
bacon-fried-rice,Bacon Fried Rice,5.000000,5.000000
chicken-fingers,Chicken Fingers,2.996816,2.996816
apple-crisp,Apple Crisp,2.211293,2.211293
...,...,...,...
bacon-mac-and-cheese,Bacon Mac And Cheese,5.000000,5.000000
chicken-alfredo-lasagna,Chicken Alfredo Lasagna,2.263525,2.263525
classic-beef-lasagna,Classic Beef Lasagna,2.659932,2.659932


In [23]:
features = recipes[[c for c in recipes.columns if not c.startswith("recipe_")]]
features

,apple,bacon,chicken,crisp,lasagna,veg
recipe_slug,,,,,,
falafel,False,False,False,False,False,False
spamburger,False,False,False,False,False,False
bacon-fried-rice,False,True,False,False,False,False
chicken-fingers,False,False,True,False,False,False
apple-crisp,True,False,False,True,False,False
...,...,...,...,...,...,...
bacon-mac-and-cheese,False,True,False,False,False,False
chicken-alfredo-lasagna,False,False,True,False,True,False
classic-beef-lasagna,False,False,False,False,True,False


## Sanity Check: $X^T X$ Calculation

In [24]:
fn = features.to_numpy()
fn.T @ fn

array([[ True, False, False,  True, False, False],
       [False,  True,  True, False, False, False],
       [False,  True,  True, False,  True, False],
       [ True, False, False,  True, False, False],
       [False, False,  True, False,  True,  True],
       [False, False, False, False,  True,  True]])

In [25]:
fnf = features.to_numpy(dtype='float64')
fnf.T @ fn

array([[ 4.,  0.,  0.,  2.,  0.,  0.],
       [ 0., 13.,  1.,  0.,  0.,  0.],
       [ 0.,  1.,  5.,  0.,  1.,  0.],
       [ 2.,  0.,  0.,  6.,  0.,  0.],
       [ 0.,  0.,  1.,  0.,  6.,  2.],
       [ 0.,  0.,  0.,  0.,  2.,  2.]])

## Sanity Check: Feature Coverage

In [26]:
features.sum()

,0
apple,4
bacon,13
chicken,5
crisp,6
lasagna,6
veg,2


In [27]:
features.T.sum().T

,0
recipe_slug,
falafel,0
spamburger,0
bacon-fried-rice,1
chicken-fingers,1
apple-crisp,2
...,...
bacon-mac-and-cheese,1
chicken-alfredo-lasagna,2
classic-beef-lasagna,1


## Calculate Bounds for LinUCB Choices

In [28]:
def calculate_bounds(recipe_choices=[], alpha=1.0):
    recipe_choices = list(recipe_choices)

    D = features.loc[recipe_choices].to_numpy(dtype="float64")
    c = ratings.loc[recipe_choices, 'rating'].to_numpy(dtype="float64")

    DTDI = np.eye(D.shape[1])
    if len(recipe_choices) > 0:
        DTDI += D.T @ D
    DTDI_inv = np.linalg.inv(DTDI)

    theta_hat = DTDI_inv @ D.T @ c
    features_array = features.to_numpy()
    means = features_array @ theta_hat

    variances = []
    for z in features_array:
        z = z.reshape(-1, 1)
        variances.append((z.T @ DTDI_inv @ z).item())
    variances = np.array(variances)

    df = pd.DataFrame({"score_estimate": means,
    "score_bound": means + alpha * np.sqrt(variances)}, index=features.index)
    df["num_features"] = features.T.sum().T

    return df

calculate_bounds().sort_values("score_bound", ascending=False)

,score_estimate,score_bound,num_features
recipe_slug,,,
cranberry-apple-crisp,0.0,1.414214,2
apple-crisp,0.0,1.414214,2
chicken-alfredo-lasagna,0.0,1.414214,2
vegetable-lasagna,0.0,1.414214,2
vegetarian-mushroom-lasagna,0.0,1.414214,2
...,...,...,...
cherry-pie,0.0,0.000000,0
cold-sesame-noodles,0.0,0.000000,0
soba-noodle-salad-with-peanut-dressing,0.0,0.000000,0


In [29]:
calculate_bounds(["bacon-mac-and-cheese"] * 3)

,score_estimate,score_bound,num_features
recipe_slug,,,
falafel,0.00,0.000000,0
spamburger,0.00,0.000000,0
bacon-fried-rice,3.75,4.250000,1
chicken-fingers,0.00,1.000000,1
apple-crisp,0.00,1.414214,2
...,...,...,...
bacon-mac-and-cheese,3.75,4.250000,1
chicken-alfredo-lasagna,0.00,1.414214,2
classic-beef-lasagna,0.00,1.000000,1


In [30]:
def try_picks(**kwargs):
    recipe_choices = []
    for i in range(100):
        current_bounds = calculate_bounds([r.index[0] for r in recipe_choices], **kwargs)
        best_bound = current_bounds["score_bound"].max()
        best_recipes = current_bounds[current_bounds["score_bound"] == best_bound].copy()
        best_recipes["true_rating"] = ratings.loc[best_recipes.index, "rating"]
        choice = best_recipes.sample(1)
        recipe_choices.append(choice)

    return pd.concat(recipe_choices, axis=0).reset_index()

picks = try_picks()
picks

,recipe_slug,score_estimate,score_bound,num_features,true_rating
0,chicken-alfredo-lasagna,0.000000,1.414214,2,2.263525
1,chicken-alfredo-lasagna,1.509017,2.325513,2,2.263525
2,chicken-alfredo-lasagna,1.810820,2.443276,2,2.263525
3,chicken-alfredo-lasagna,1.940164,2.474687,2,2.263525
4,chicken-alfredo-lasagna,2.012022,2.483427,2,2.263525
...,...,...,...,...,...
95,chicken-alfredo-lasagna,2.251674,2.354003,2,2.263525
96,chicken-alfredo-lasagna,2.251797,2.353594,2,2.263525
97,chicken-alfredo-lasagna,2.251917,2.353191,2,2.263525
98,chicken-alfredo-lasagna,2.252035,2.352794,2,2.263525


In [31]:
picks.groupby("recipe_slug").count()

,score_estimate,score_bound,num_features,true_rating
recipe_slug,,,,
chicken-alfredo-lasagna,100,100,100,100


In [32]:
picks = try_picks(alpha=2)
picks.groupby("recipe_slug").count()

,score_estimate,score_bound,num_features,true_rating
recipe_slug,,,,
apple-crisp,4,4,4,4
bacon-wrapped-chicken,92,92,92,92
chicken-alfredo-lasagna,1,1,1,1
cranberry-apple-crisp,2,2,2,2
vegetarian-mushroom-lasagna,1,1,1,1


In [34]:
ratings["rating_good"] = (ratings["rating_bad"] - 1) * 0.25
ratings["rating"] = ratings["rating_good"]
ratings

,recipe_title,rating_bad,rating,rating_good
recipe_slug,,,,
falafel,Falafel,2.809655,0.452414,0.452414
spamburger,Spamburger,2.304000,0.326000,0.326000
bacon-fried-rice,Bacon Fried Rice,5.000000,1.000000,1.000000
chicken-fingers,Chicken Fingers,2.996816,0.499204,0.499204
apple-crisp,Apple Crisp,2.211293,0.302823,0.302823
...,...,...,...,...
bacon-mac-and-cheese,Bacon Mac And Cheese,5.000000,1.000000,1.000000
chicken-alfredo-lasagna,Chicken Alfredo Lasagna,2.263525,0.315881,0.315881
classic-beef-lasagna,Classic Beef Lasagna,2.659932,0.414983,0.414983


In [41]:
picks = try_picks(alpha=3)
picks.groupby("recipe_slug").count()

,score_estimate,score_bound,num_features,true_rating
recipe_slug,,,,
apple-crisp,5,5,5,5
apple-crumble,1,1,1,1
apple-pie,3,3,3,3
bacon-and-egg-breakfast-sandwich,3,3,3,3
bacon-chocolate-chip-cookies,3,3,3,3
bacon-egg-muffins,5,5,5,5
bacon-fried-rice,4,4,4,4
bacon-mac-and-cheese,1,1,1,1
bacon-souffle,4,4,4,4


In [40]:
picks

,recipe_slug,score_estimate,score_bound,num_features,true_rating
0,cranberry-apple-crisp,0.000000,2.828427,2,0.294693
1,vegetarian-mushroom-lasagna,0.000000,2.828427,2,0.000000
2,bacon-wrapped-chicken,0.000000,2.828427,2,1.000000
3,chicken-alfredo-lasagna,0.333333,2.642734,2,0.315881
4,bacon-wrapped-chicken,0.664174,2.237765,2,1.000000
...,...,...,...,...,...
95,bacon-souffle,0.967139,1.288071,1,1.000000
96,bacon-wrapped-chicken,1.004528,1.287888,2,1.000000
97,bacon-wrapped-chicken,1.004439,1.284997,2,1.000000
98,bacon-souffle,0.967956,1.284822,1,1.000000
